# Session 2: converting numbers, and gates that decide
**A. Real-world problem.** A stairwell light has a switch at the top and one at the bottom.
Either switch must toggle the light. A lift must refuse to move unless the door is closed AND
the load is allowed. Such rules are built from *logic gates*, tiny circuits whose output is 0 or
1 and depends only on their inputs. Before that, we need to move numbers between the three
writing systems engineers use: binary (base 2), decimal (base 10) and hexadecimal (base 16).

**Plan.** (1) write converters and test them on thousands of numbers, (2) define the gates and
print their truth tables, (3) verify De Morgan's laws by trying every input, (4) build NOT, AND,
OR and XOR from NAND alone, (5) design a majority gate and a comparator from a truth table.
This is a simplified educational model of digital logic (ideal gates, no electrical detail).

In [ ]:
import itertools
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

DIGITS = "0123456789ABCDEF"

## Step 1: place value in any base
In base b the digit string d_k ... d_1 d_0 means d_k * b^k + ... + d_1 * b + d_0.
To go the other way, divide by b again and again and collect the remainders (last one first).
Hexadecimal uses the digits 0-9 and A-F (A = 10 ... F = 15); one hex digit is exactly 4 bits.

In [ ]:
def to_base(n, base):
    """Whole number n >= 0 as a digit string in the given base (2 to 16)."""
    if n < 0:
        raise ValueError("only whole numbers from 0 up")
    if n == 0:
        return "0"
    out = []
    while n:
        n, r = divmod(n, base)
        out.append(DIGITS[r])
    return "".join(reversed(out))


def from_base(text, base):
    """Digit string in the given base back to a whole number (place values added up)."""
    value = 0
    for ch in text.upper():
        d = DIGITS.index(ch)
        if d >= base:
            raise ValueError(f"digit {ch} is not allowed in base {base}")
        value = value * base + d
    return value


def bin_to_hex(bits):
    """Group the bits in fours from the right; each group is one hex digit."""
    bits = bits.zfill(-(-len(bits) // 4) * 4)
    return "".join(to_base(from_base(bits[i:i + 4], 2), 16) for i in range(0, len(bits), 4))

## Step 2: the gates
A gate takes bits and returns a bit. We write them with numpy so one call handles many
inputs at once (arrays of 0s and 1s).

    AND  : 1 only if both inputs are 1       OR   : 1 if at least one input is 1
    NOT  : flips the bit                      XOR  : 1 if the inputs are different
    NAND : NOT of AND (0 only if both are 1)

In [ ]:
def AND(a, b):
    return np.asarray(a) & np.asarray(b)


def OR(a, b):
    return np.asarray(a) | np.asarray(b)


def NOT(a):
    return 1 - np.asarray(a)


def XOR(a, b):
    return np.asarray(a) ^ np.asarray(b)


def NAND(a, b):
    return 1 - (np.asarray(a) & np.asarray(b))


def all_inputs(n):
    """Every combination of n input bits: shape (2**n, n), first column is the biggest place."""
    return np.array(list(itertools.product((0, 1), repeat=n)), dtype=int)


def truth_table(gate, n=2):
    """Rows of inputs followed by the output of the gate on that row."""
    X = all_inputs(n)
    return np.column_stack([X, gate(*X.T)])

## Step 3: De Morgan's laws, proved by trying everything
With two inputs there are only 4 combinations, with three only 8. A claim that holds for every
row of the truth table is *proved*, not just tested. De Morgan:
NOT(a AND b) = (NOT a) OR (NOT b)  and  NOT(a OR b) = (NOT a) AND (NOT b).

In [ ]:
def de_morgan_holds(n_inputs=2):
    """True if both De Morgan laws hold on every row of the truth table (any number of inputs)."""
    X = all_inputs(n_inputs).T
    all_and, all_or = X[0], X[0]
    or_of_nots, and_of_nots = NOT(X[0]), NOT(X[0])
    for row in X[1:]:
        all_and, all_or = AND(all_and, row), OR(all_or, row)
        or_of_nots, and_of_nots = OR(or_of_nots, NOT(row)), AND(and_of_nots, NOT(row))
    law1 = np.array_equal(NOT(all_and), or_of_nots)    # NOT(a AND b ...) = NOT a OR NOT b ...
    law2 = np.array_equal(NOT(all_or), and_of_nots)    # NOT(a OR b ...) = NOT a AND NOT b ...
    return bool(law1 and law2)

## Step 4: NAND alone is enough
NOT a = NAND(a, a). AND = NOT(NAND). OR(a, b) = NAND(NOT a, NOT b) (De Morgan again).
XOR needs four NAND gates. `nand` is a parameter so that a checkpoint can count the gates used.

In [ ]:
def not_n(a, nand=NAND):
    return nand(a, a)


def and_n(a, b, nand=NAND):
    return not_n(nand(a, b), nand)


def or_n(a, b, nand=NAND):
    return nand(not_n(a, nand), not_n(b, nand))


def xor_n(a, b, nand=NAND):
    m = nand(a, b)
    return nand(nand(a, m), nand(b, m))


def counting_nand():
    """A NAND gate that counts how many times it is used: returns (gate, counter_list)."""
    counter = [0]

    def gate(a, b):
        counter[0] += 1
        return NAND(a, b)

    return gate, counter

## Step 5: designing a gate from a truth table (sum of products)
Write the wanted output for every input row. Each row with output 1 becomes one AND term
(a *minterm*) that is true for exactly that row; OR the terms together. This always works.
Example 1: a 3-input *majority* gate (1 when at least two inputs are 1).
Example 2: a 2-bit comparator, output 1 when the number A = (a1 a0) is greater than B = (b1 b0).

In [ ]:
def minterms(outputs):
    """Row numbers (0 = all inputs 0) where the wanted output is 1."""
    return [i for i, o in enumerate(outputs) if o == 1]


def sop_eval(terms, n, X):
    """Evaluate the sum of products on rows X (shape (rows, n)): OR over minterms of an AND of literals."""
    X = np.asarray(X)
    out = np.zeros(len(X), dtype=int)
    for t in terms:
        bits = [(t >> (n - 1 - k)) & 1 for k in range(n)]
        term = np.ones(len(X), dtype=int)
        for k, want in enumerate(bits):
            term = AND(term, X[:, k] if want else NOT(X[:, k]))
        out = OR(out, term)
    return out


def majority_outputs():
    return [1 if sum(row) >= 2 else 0 for row in all_inputs(3)]


def greater_outputs():
    """Rows are (a1, a0, b1, b0); output 1 when 2*a1 + a0 > 2*b1 + b0."""
    return [1 if 2 * r[0] + r[1] > 2 * r[2] + r[3] else 0 for r in all_inputs(4)]


def majority_short(a, b, c):
    """The simplified form a.b + a.c + b.c."""
    return OR(OR(AND(a, b), AND(a, c)), AND(b, c))


def greater_short(a1, a0, b1, b0):
    """Simplified: a1.(NOT b1) + (a1 XNOR b1).a0.(NOT b0)."""
    same = NOT(XOR(a1, b1))
    return OR(AND(a1, NOT(b1)), AND(AND(same, a0), NOT(b0)))

## Step 6: see it
Left: the truth tables of four gates as a grid. Right: the comparator, with A down the side and
B along the top, 1 where A > B.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    for n in (5, 10, 183, 255, 4095):
        print(f"{n:5d} = {to_base(n, 2)} (binary) = {to_base(n, 16)} (hex)")
    gates = [("AND", AND), ("OR", OR), ("XOR", XOR), ("NAND", NAND)]
    fig, ax = plt.subplots(figsize=(6, 3.4))
    grid = np.array([[g(a, b) for g_name, g in gates] for a, b in all_inputs(2)])
    ax.imshow(grid, cmap="YlOrBr", vmin=0, vmax=1.4, aspect="auto")
    for (r, c), v in np.ndenumerate(grid):
        ax.text(c, r, str(v), ha="center", va="center")
    ax.set_xticks(range(4), [g[0] for g in gates])
    ax.set_yticks(range(4), [f"a={a} b={b}" for a, b in all_inputs(2)])
    ax.set_xlabel("gate")
    ax.set_ylabel("inputs")
    ax.set_title("Truth tables: output of each gate (1 = lamp on)")
    fig.tight_layout()
    fig.savefig(OUT / "gate_truth_tables.png", dpi=120)

    cmp = np.zeros((4, 4), dtype=int)
    for A in range(4):
        for B in range(4):
            cmp[A, B] = greater_short(A >> 1, A & 1, B >> 1, B & 1)
    fig2, ax2 = plt.subplots(figsize=(4.4, 4))
    ax2.imshow(cmp, cmap="YlOrBr", vmin=0, vmax=1.4)
    for (r, c), v in np.ndenumerate(cmp):
        ax2.text(c, r, str(v), ha="center", va="center")
    ax2.set_xticks(range(4))
    ax2.set_yticks(range(4))
    ax2.set_xlabel("B (decimal value of b1 b0)")
    ax2.set_ylabel("A (decimal value of a1 a0)")
    ax2.set_title("Comparator output: 1 where A > B")
    fig2.tight_layout()
    fig2.savefig(OUT / "comparator.png", dpi=120)
    print("majority minterm rows:", minterms(majority_outputs()))
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print `to_base(n, 2)` and `to_base(n, 16)` for n = 255, 256 and 4095. Predict each by hand first.
2. **Stretch.** Write `nor_n(a, b)` using only NAND gates (NOR is NOT of OR) and check it on all four input rows. How many NAND gates did you use?
3. **Explain.** Design a 3-input odd-parity gate (output 1 when an odd number of inputs are 1). Write its sum of products with `minterms` and verify it on all 8 rows.

## Assessment
- **A. Conceptual.** Why can NAND build every other gate? Show how to get NOT and OR from NAND.
- **B. Numerical.** Convert the hexadecimal number A5 to decimal and to binary, and convert decimal 100 to binary and hexadecimal. Sanity check.
- **C. Graph and simulation interpretation.** In the comparator figure (A down the side, B along the top, dark where A > B), where are the dark squares and how many are there? Why is the diagonal white?
- **D. Design and optimisation challenge.** Design: a lock opens when the 2-bit number A equals the 2-bit number B. Build the truth table, write the sum-of-products form and give a shorter form with XNOR. How many minterms are there?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []

    # converters against Python's own
    ok_bin = all(L.to_base(n, 2) == bin(n)[2:] and L.from_base(bin(n)[2:], 2) == n for n in range(5000))
    ok_hex = all(L.to_base(n, 16) == format(n, "X") and L.from_base(format(n, "x"), 16) == n for n in range(5000))
    ok_dec = all(L.to_base(n, 10) == str(n) for n in range(5000))
    checks.append(("converters agree with Python for every number 0 to 4999 (binary, decimal, hex)", ok_bin and ok_hex and ok_dec))
    checks.append(("round trip in every base 2 to 16 for 0 to 600",
                   all(L.from_base(L.to_base(n, b), b) == n for b in range(2, 17) for n in range(601))))
    checks.append(("worked example: 183 is 10110111 in binary and B7 in hex, and 128+32+16+4+2+1 = 183",
                   L.to_base(183, 2) == "10110111" and L.to_base(183, 16) == "B7"
                   and L.from_base("B7", 16) == 183 and 128 + 32 + 16 + 4 + 2 + 1 == 183
                   and L.bin_to_hex("10110111") == "B7" and 11 * 16 + 7 == 183))
    checks.append(("edge: zero is '0'; 4095 is FFF; digit 2 is not allowed in binary",
                   L.to_base(0, 2) == "0" and L.to_base(4095, 16) == "FFF" and _raises(L.from_base, "102", 2)))

    # gates and truth tables
    tt = {name: L.truth_table(g)[:, 2].tolist() for name, g in
          (("AND", L.AND), ("OR", L.OR), ("XOR", L.XOR), ("NAND", L.NAND))}
    checks.append(("truth tables of AND, OR, XOR, NAND (rows 00, 01, 10, 11)",
                   tt == {"AND": [0, 0, 0, 1], "OR": [0, 1, 1, 1], "XOR": [0, 1, 1, 0], "NAND": [1, 1, 1, 0]}))
    checks.append(("NOT flips 0 and 1", L.NOT(0) == 1 and L.NOT(1) == 0))

    # De Morgan, exhaustive
    checks.append(("De Morgan holds on all 4 rows (2 inputs) and all 8 rows (3 inputs)",
                   L.de_morgan_holds(2) and L.de_morgan_holds(3) and len(L.all_inputs(2)) == 4 and len(L.all_inputs(3)) == 8))
    # a law that is NOT true must be caught: NOT(a AND b) is not NOT a AND NOT b
    X = L.all_inputs(2).T
    wrong = np.array_equal(L.NOT(L.AND(X[0], X[1])), L.AND(L.NOT(X[0]), L.NOT(X[1])))
    checks.append(("a wrong law is detected: NOT(a AND b) differs from (NOT a) AND (NOT b)", not wrong))

    # NAND universality and gate counts
    X2 = L.all_inputs(2)
    a, b = X2[:, 0], X2[:, 1]
    checks.append(("NOT, AND, OR, XOR built only from NAND match the real gates on every row",
                   np.array_equal(L.not_n(a), L.NOT(a)) and np.array_equal(L.and_n(a, b), L.AND(a, b))
                   and np.array_equal(L.or_n(a, b), L.OR(a, b)) and np.array_equal(L.xor_n(a, b), L.XOR(a, b))))
    used = []
    for fn, args in ((L.not_n, (a,)), (L.and_n, (a, b)), (L.or_n, (a, b)), (L.xor_n, (a, b))):
        g, counter = L.counting_nand()
        fn(*args, nand=g)
        used.append(counter[0])
    checks.append(("NAND gates needed: NOT 1, AND 2, OR 3, XOR 4", used == [1, 2, 3, 4]))

    # design problems
    X3 = L.all_inputs(3)
    maj = L.majority_outputs()
    checks.append(("majority gate has minterm rows 3, 5, 6, 7", L.minterms(maj) == [3, 5, 6, 7]))
    checks.append(("majority: sum of products matches the table on all 8 rows",
                   L.sop_eval(L.minterms(maj), 3, X3).tolist() == maj))
    checks.append(("majority: short form a.b + a.c + b.c matches on all 8 rows",
                   L.majority_short(X3[:, 0], X3[:, 1], X3[:, 2]).tolist() == maj))
    X4 = L.all_inputs(4)
    gt = L.greater_outputs()
    checks.append(("comparator A > B has 6 minterms of 16 (pairs with A greater: 3+2+1)",
                   len(L.minterms(gt)) == 6 and 3 + 2 + 1 == 6))
    checks.append(("comparator: sum of products matches A > B on all 16 rows",
                   L.sop_eval(L.minterms(gt), 4, X4).tolist() == gt))
    checks.append(("comparator: short form matches A > B on all 16 rows",
                   L.greater_short(*X4.T).tolist() == gt))
    # equality comparator (assessment D): 4 minterms, equals NOT(XOR) on each bit pair
    eq = [1 if 2 * r[0] + r[1] == 2 * r[2] + r[3] else 0 for r in X4]
    xnor_form = L.AND(L.NOT(L.XOR(X4[:, 0], X4[:, 2])), L.NOT(L.XOR(X4[:, 1], X4[:, 3])))
    checks.append(("equality comparator has 4 minterms and equals (a1 XNOR b1) AND (a0 XNOR b0)",
                   len(L.minterms(eq)) == 4 and xnor_form.tolist() == eq))
    checks.append(("A5 is 165 and 10100101; 100 is 1100100 and 64 in hex; one hex digit is four bits",
                   L.from_base("A5", 16) == 165 and L.to_base(165, 2) == "10100101" and 10 * 16 + 5 == 165
                   and L.to_base(100, 2) == "1100100" and L.to_base(100, 16) == "64" and 6 * 16 + 4 == 100
                   and L.bin_to_hex("1111") == "F" and 128 + 32 + 4 + 1 == 165))
    grid = np.array(L.greater_outputs()).reshape(4, 4)          # grid[A, B]
    checks.append(("comparator figure: the six ones lie strictly below the diagonal, the diagonal is 0",
                   np.array_equal(grid, np.tril(np.ones((4, 4), dtype=int), -1))))
    checks.append(("equality rows are 0000, 0101, 1010, 1111",
                   [format(i, "04b") for i in L.minterms(eq)] == ["0000", "0101", "1010", "1111"]))
    return checks


def _raises(fn, *args):
    try:
        fn(*args)
    except ValueError:
        return True
    return False

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
